# Analisis con Pandas y Kaggle

**Preparacion del entorno**
Primero debemos preparar el entorno de trabajo con Pandas

**Cargar los datos**
Una vez seleccionado y descargado el dataset de Kaggle se realiza la carga del archivo csv en un DataFrame de Pandas


In [ ]:
import pandas as pd

df = pd.read_csv("/Users/carlis/mi_proyecto/Pandas y Visualizacion/supermarket_sales 2.csv")



**Exploracion inicial de los datos**
En esta parte comenzamos a revisar y explorar la informacion del DataFrame utilizando las operaciones basicas como Head, Tail, Info, Describe y Types.

Con Head y Tail podemos visualizar las primeras y ultimas filas respectivamente
Con Info entrega un resumen conciso del DataFrame
Con Describe "genera estadisticas descriptivas que resumen la tendencia central, dispersion y forma de la distribucion de un conjunto de datos"
Con Types nos permite saber los tipos de datos de cada columna

In [5]:
print("Primeras filas del DataFrame:")
print(df.head(10))

print("Ultimas filas del DataFrame:")
print(df.tail(5))

print("Informacion del DataFrame:")
print(df.info())

print("Estadisticas descriptivas del DataFrame:")
print(df.describe().round(2))

print("Tipos originales:")
print(df.dtypes)

Primeras filas del DataFrame:
    Invoice ID Branch       City Customer type  Gender  \
0  750-67-8428      A     Yangon        Member  Female   
1  226-31-3081      C  Naypyitaw        Normal  Female   
2  631-41-3108      A     Yangon        Normal    Male   
3  123-19-1176      A     Yangon        Member    Male   
4  373-73-7910      A     Yangon        Normal    Male   
5  699-14-3026      C  Naypyitaw        Normal    Male   
6  355-53-5943      A     Yangon        Member  Female   
7  315-22-5665      C  Naypyitaw        Normal  Female   
8  665-32-9167      A     Yangon        Member  Female   
9  692-92-5582      B   Mandalay        Member  Female   

             Product line  Unit price  Quantity   Tax 5%     Total       Date  \
0       Health and beauty       74.69         7  26.1415  548.9715   1/5/2019   
1  Electronic accessories       15.28         5   3.8200   80.2200   3/8/2019   
2      Home and lifestyle       46.33         7  16.2155  340.5255   3/3/2019   
3      

**Limpieza de datos**
En este punto revisamos los tipos de datos de cada columna, si tenemos datos nulos y/o duplicados

Para el DataFrame seleccionado revisamos lo siguiente:
Primero se confirma si tenemos datos nulos y logramos verificar que este archivo no contiene datos nulos
Segundo, se revisa el tipo de datos de la columnas y nos encontramos con que tenemos la columna Date con un tipo de dato str y consideramos que para un mejor analisis esta informacion debe esta en tipo datetime ya que asi podriamos tomar esta columna y obtener informacion de las ventas segun el mes o año. Ademas se revisa que la columna Quantity se encuentra en tipo float (decimales) y se realiza el cambio a tipo int (numero entero)
Tercero, se realiza la revision para verificar si tenemos filas duplicadas, para este caso nos indica que no hay filas duplicadas, luego se decide revisar si la columna Payment tiene informacion duplicada y nos indica que existen duplicados lo que es correcto ya que en esta columna podemos confirmar como realizo el pago cada cliente, ya sea efectivo, tarjeta y otros.


In [7]:
print("Verificacion de datos nulos:")
print(df.isnull().sum())

print("Cantidad de nulos en la columna Date:")
print(df['Date'].isnull().sum())


#en este caso se visualiza que la fecha esta en formato str segun lo revisdo en la red esto estaria bien al venir de un archivo csv pero para analizar datos por este medio ese formato no seria de ayuda por lo tanto se realiza este cambio
df['Date'] = pd.to_datetime(df['Date'], format="%m/%d/%Y")
print("Fecha convertida:")
print(df[['Date']].dtypes)

#Otro caso que nos interesa corregir sera la columna Quantity ya que se encuentra en float osea numero decimal y consideramos que para un mejor analisis es necesario que se cambie a numero entero
df['Quantity'] = df['Quantity'].astype(int)
print("Cantidad convertida:")
print(df[['Quantity']].dtypes)

#Confirmar si existen filas duplicadas
n_duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {n_duplicados}")

col_duplicados = df.duplicated(subset=['Payment']).sum()
print(f"Columna payment duplicada: {col_duplicados}")
# para este caso se entiende que este bien que existan datos duplicados en la columna payment ya que esta columna indica el metodo de pago


Verificacion de datos nulos:
Invoice ID                 0
Branch                     0
City                       0
Customer type              0
Gender                     0
Product line               0
Unit price                 0
Quantity                   0
Tax 5%                     0
Total                      0
Date                       0
Time                       0
Payment                    0
cogs                       0
gross margin percentage    0
gross income               0
Rating                     0
dtype: int64
Cantidad de nulos en la columna Date:
0
Fecha convertida:
Date    datetime64[us]
dtype: object
Cantidad convertida:
Quantity    int64
dtype: object
Filas duplicadas: 0
Columna payment duplicada: 997


**Transformacion de datos**

1.Crear nuevas columnas, ya que el dataset seleccionado viene con bastante informacion se decide realizar una comprobacion del valor neto para verificar si los montos son correctos y con esto podriamos tambien calcular un promedio por unidad.

2.Normalizar, se realiza la normalizacion del total para escalar los valores en un rango especifico.

3.Clasificar datos, se decide realizar una clasificacion de las ventas, si el total es mayor a 500 se considera venta alta, si el valor es menor se considera venta baja.

In [3]:
#Comprobacion valor neto
df['Comprobacion_valor_neto'] = df['Unit price'] * df['Quantity']
print("Nueva Columna:")
df[['cogs', 'Comprobacion_valor_neto']].head()

Nueva Columna:


,cogs,Comprobacion_valor_neto
0,522.83,522.83
1,76.40,76.40
2,324.31,324.31
3,465.76,465.76
4,604.17,604.17


In [4]:
#Venta promedio por unidad
df['promedio_unidad'] = df['cogs'] / df['Quantity']
print('Promedio por unidad:')
df[['Comprobacion_valor_neto', 'promedio_unidad']].head()

,Comprobacion_valor_neto,promedio_unidad
0,522.83,74.69
1,76.40,15.28
2,324.31,46.33
3,465.76,58.22
4,604.17,86.31


In [5]:
#normalizar columna total
max_value = df['Total'].max()
min_value = df['Total'].min()
df['Total_normalizado'] = df['Total'].apply(lambda x: (x - min_value)/(max_value - min_value))
print("Nueva columna 'Total_normalizado':")
df[['Total', 'Total_normalizado']].head()


Nueva columna 'Total_normalizado':


,Total,Total_normalizado
0,548.9715,0.521616
1,80.2200,0.067387
2,340.5255,0.319628
3,489.0480,0.463549
4,634.3785,0.604377


In [8]:
#Clasificar datos
df['Clasificacion'] = df['Total'].apply(lambda x: 'Alta' if x >= 500 else 'Baja')
print('Clasificacion:')
df[['Total', 'Clasificacion']].head()


Clasificacion:


,Total,Clasificacion
0,548.9715,Alta
1,80.2200,Baja
2,340.5255,Baja
3,489.0480,Baja
4,634.3785,Alta


**Analisis de datos**

1.Primero realizaremos agrupaciones de datos con groupby donde podremos ver total de ventas por linea de producto, total de ventas por sucursal y cantidad de ventas por sucursal.

2.En este punto utilizaremos la funcion de agregacion y podremos ver un detalle por sucursal de la suma de sus ventas, un promedio y la cantidad  de ventas.

3.En este ultimo punto utilizaremos la funcion apply, continuando con la informacion de las sucursales ahora calcularemos el porcentaje de ventas por sucursal.

In [10]:
#Agrupaciones con Groupby
ventas_por_linea_producto = df.groupby('Product line')['Total'].sum()
print("Ventas por linea de producto:")
print(ventas_por_linea_producto)


ventas_por_tienda = df.groupby('Branch')['Total'].sum()
print("Ventas por tienda:")
print(ventas_por_tienda)

grouped = df.groupby('Branch')
cantidad_ventas_branch = grouped['Total'].count()
print('Cantidad de ventas por sucursal')
print(cantidad_ventas_branch)

Ventas por linea de producto:
Product line
Electronic accessories    54337.5315
Fashion accessories       54305.8950
Food and beverages        56144.8440
Health and beauty         49193.7390
Home and lifestyle        53861.9130
Sports and travel         55122.8265
Name: Total, dtype: float64
Ventas por tienda:
Branch
A    106200.3705
B    106197.6720
C    110568.7065
Name: Total, dtype: float64


In [ ]:
#Funcion de agregacion
agregacion_branch = df.groupby('Branch')['Total'].agg(['sum', 'mean', 'count'])
print("Resumen por sucursal:")
print(agregacion_branch)

Branch
A    340
B    332
C    328
Name: Total, dtype: int64
                sum        mean  count
Branch                                
A       106200.3705  312.354031    340
B       106197.6720  319.872506    332
C       110568.7065  337.099715    328


In [ ]:
#Utilizacion metodo apply
total_global = df['Total'].sum()

porcentaje_branch = ventas_por_tienda.apply(lambda x: (x / total_global) * 100)
print('Porcentaje de ventas por sucursal:')
print(porcentaje_branch)